# Stage C1 — Physics Constraints

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Sec. 3.3 (Table 7, Eq. 3.7 and 3.9–3.15).

**Physics starts here.** B1/B2 were deliberately physics-free; this is
where the six constraint penalties get implemented and — just as
important — unit-tested against functions with a **known** derivative
sign, so a sign bug gets caught in seconds here instead of after a
multi-hour training run in C4. The unit tests stop the notebook if any
of them fails, so a batch run cannot continue past a broken constraint.

**Input:** `data/masters_data.xlsx` (for the density/correlation
reference used by Section 8's adaptive weighting and for the
normalization constants — the constraints themselves are evaluated on
collocation points, not training data), `outputs/B1_selected_architecture.json`
(architecture of the Section 9 wiring check).
**Output:** every figure and result table is saved to `outputs/html/` as
`C1_<section>[_qualifier].html`. For C2 (read by code, so not HTML):
`outputs/C1_collocation_points.csv` (1000 LHS points + density ratio +
validity) and `outputs/C1_constraint_config.json` (non-negativity floors,
Eq. 3.9 direction, validity rule).

**Framework note:** the constraints differentiate a model's output
with respect to its *input*, which needs `tf.GradientTape` — not
something `polars`/`plotly` can do. One of the six needs a **second**
derivative (nested tapes) and two need **two separate** gradients from
the same tape (`persistent=True`) — both flagged explicitly below,
since they're the two easiest places to introduce a silent bug.

**Changes from the first version of this notebook** (details in each
section): non-negativity is now measured against the *physical* zero,
not the normalized zero (Section 7); the η–NOx validity function follows
the sign rule of Sec. 3.3.2.2 (Section 8); the Eq. 3.9 direction is a
single named setting (Section 4); the wiring check uses the same model
builder as B1/B2, including the η output head.

## Setup

In [ ]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from scipy.stats import qmc, gaussian_kde
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42
RAW_PATH


## Color palette and output naming (shared across the whole pipeline)

`SPLIT_COLORS` (semantic role: train / validation / test / reference
value / alert / neutral) and `VARIABLE_COLORS` (identity of each of the
4 inputs and 5 outputs) are identical in every A/B/C notebook, so the
same element always has the same color in any chart of the pipeline.

Every figure or result table generated below is also saved to
`outputs/html/`, named `C1_<section>[_qualifier].html` — the number
matches the corresponding section header, so the order in which each
output was produced can be read from the file name alone.

In [ ]:
SPLIT_COLORS = {
    "train": "#B7C9DA",
    "validation": "#2B6EFF",
    "test": "#571D99",
    "reference": "#343A40",   # value transcribed from the dissertation text
    "alert": "#E85D04",       # outlier / out of range / anomaly
    "neutral": "#B0AFA8",     # grid lines / neutral reference
}
VARIABLE_COLORS = {
    "SOI": "#073b3a", "lambda": "#0b6e4f", "sub_rate": "#08a045", "P_rail": "#6bbf59",
    "NOx": "#c7adff", "PM": "#916dd5", "eta": "#7151a9", "HC": "#573d7f", "CO2": "#46325d",
}

HTML_DIR = OUT_DIR / "html"
HTML_DIR.mkdir(parents=True, exist_ok=True)


def flagged_table_html(df, title, out_path, flag_col=None, is_flagged=lambda v: False, ref_cols=()):
    """Result table -> Plotly go.Table -> HTML.
    Columns listed in ref_cols get the 'reference' tone in the header
    (values transcribed from the dissertation text). Cells in flag_col
    get the 'alert' tone wherever is_flagged(value) is True."""
    cols = list(df.columns)
    n = df.shape[0]
    header_fill = [SPLIT_COLORS["reference"] if c in ref_cols else "#F1F3F5" for c in cols]
    header_font = ["white" if c in ref_cols else "black" for c in cols]
    cell_fill = []
    for c in cols:
        if c == flag_col:
            cell_fill.append([SPLIT_COLORS["alert"] if is_flagged(v) else "white"
                               for v in df[c].to_list()])
        else:
            cell_fill.append(["white"] * n)
    fig = go.Figure(data=[go.Table(
        header=dict(values=cols, fill_color=header_fill,
                     font=dict(color=header_font), align="left"),
        cells=dict(values=[df[c].to_list() for c in cols],
                    fill_color=cell_fill, align="left"),
    )])
    fig.update_layout(title=title, margin=dict(t=40, l=10, r=10, b=10))
    fig.write_html(str(out_path), include_plotlyjs="inline")
    return fig


def simple_table_html(df, title, out_path):
    return flagged_table_html(df, title, out_path)

## 1. Load & normalize (same logic as A3/B1/B2)

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)

# index constants, used throughout instead of magic numbers
SOI_IDX, LAMBDA_IDX, SUBRATE_IDX, PRAIL_IDX = [INPUT_COLS.index(c) for c in INPUT_COLS]
HC_IDX, NOX_IDX, CO2_IDX, PM_IDX, ETA_IDX = [OUTPUT_COLS.index(c) for c in OUTPUT_COLS]
EMISSION_IDXS = [HC_IDX, NOX_IDX, CO2_IDX, PM_IDX]  # eta excluded -- bounded by sigmoid instead, Sec 3.3.3.1

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]

medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2
rng_np = np.random.default_rng(SEED)
jitter = rng_np.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))

train_df = df.filter(pl.col("split") == "train")
train_min = {c: train_df[c].min() for c in ALL_COLS}
train_max = {c: train_df[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])
X_train = df.filter(pl.col("split") == "train").select([f"{c}_norm" for c in INPUT_COLS]).to_numpy()
print("train inputs for the density/correlation reference:", X_train.shape)


## 2. Latin Hypercube collocation sampling

1000 points across the normalized 4D input domain $[0,1]^4$ — this is
where the physics constraints get evaluated (Sec. 3.2.2.2), everywhere
in the domain, not only at the 40 experimental points.

**Domain note, to decide before C4:** $[0,1]^4$ is the *train-derived*
range (A3, Section 5). The validation/test inputs reach roughly −0.4 to
1.35 in normalized units (A3, Section 6), so the constraints are not
enforced in the extrapolation region the test set probes — which is
where physics priors are expected to help most. Extending the LHS box to
the input ranges of the full test matrix (inputs only, no targets) is an
option; this notebook keeps the text's formulation.

In [ ]:
N_COLLOC = 1000
sampler = qmc.LatinHypercube(d=N_IN, seed=SEED)
X_colloc = sampler.random(n=N_COLLOC).astype(np.float32)
print(X_colloc.shape, X_colloc.min(), X_colloc.max())


**Coverage check** — pairwise projections of the 1000 collocation points against the 28 training points:

In [ ]:
fig = make_subplots(rows=2, cols=3, subplot_titles=[
    f"{a} vs {b}" for i, a in enumerate(INPUT_COLS) for b in INPUT_COLS[i+1:]
])
pairs = [(i, j) for i in range(N_IN) for j in range(i + 1, N_IN)]
for k, (i, j) in enumerate(pairs):
    r, c = divmod(k, 3)
    fig.add_trace(go.Scatter(x=X_colloc[:, i], y=X_colloc[:, j], mode="markers",
                              marker=dict(color=SPLIT_COLORS["neutral"], size=4, opacity=0.6),
                              name="collocation points", showlegend=(k == 0)), row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=X_train[:, i], y=X_train[:, j], mode="markers",
                              marker=dict(color=SPLIT_COLORS["train"], size=9,
                                          line=dict(color=SPLIT_COLORS["reference"], width=1)),
                              name="train points", showlegend=(k == 0)), row=r + 1, col=c + 1)
    fig.update_xaxes(title_text=f"{INPUT_COLS[i]} (norm)", row=r + 1, col=c + 1)
    fig.update_yaxes(title_text=f"{INPUT_COLS[j]} (norm)", row=r + 1, col=c + 1)
fig.update_layout(height=600, width=1000, title_text="LHS collocation coverage vs. training data (normalized)")
fig.show()
fig.write_html(str(HTML_DIR / "C1_02_collocation_coverage.html"), include_plotlyjs="inline")

## 3. Synthetic test harness

Builds a fake 5-output "model" from simple formulas, so each
constraint can be checked against a case with a **known** correct sign
before it ever sees a real network. `formulas` maps an output index to
a function of `x`; any output not listed is filled with zeros (its
gradient is irrelevant to the test at hand).

Every case is recorded with its expected result (`zero` = the penalty
must vanish, `positive` = it must be clearly above zero); Section 7 ends
with the full table and stops the notebook if any case fails.

In [ ]:
def synthetic_output(x, formulas):
    cols = []
    for i in range(N_OUT):
        cols.append(formulas[i](x) if i in formulas else tf.zeros(tf.shape(x)[0]))
    return tf.stack(cols, axis=1)

UNIT_TESTS = []

def run_case(constraint_fn, predict_fn, x, label, expect, constraint_name):
    penalty = float(constraint_fn(predict_fn, x))
    passed = penalty < 1e-8 if expect == "zero" else penalty > 1e-6
    UNIT_TESTS.append({"constraint": constraint_name, "case": label, "expected": expect,
                       "penalty": penalty, "passed": passed})
    print(f"  {label:34s} penalty = {penalty:.6f}   expected {expect:8s} -> {'PASS' if passed else 'FAIL'}")
    return penalty

X_test_pts = tf.constant(np.random.default_rng(0).uniform(0, 1, (100, N_IN)), dtype=tf.float32)

## 4. Monotonic constraints — NOx vs. SOI (Eq. 3.9) and PM vs. λ (Eq. 3.10)

$$
\mathcal{L}_{\text{NOx-SOI}} = \frac{1}{N_c}\sum_{i=1}^{N_c} \max\!\left(0,\ \frac{\partial \text{NOx}}{\partial \text{SOI}}\Big|_i\right)^{2}
\qquad
\mathcal{L}_{\text{PM-}\lambda} = \frac{1}{N_c}\sum_{i=1}^{N_c} \max\!\left(0,\ \frac{\partial \text{PM}}{\partial \lambda}\Big|_i\right)^{2}
$$

Both are written in the text as **decreasing** relationships, so
positive derivatives get penalized — same formula, different (output,
input) pair, implemented once and reused. The function takes the
direction as an argument so the other direction is one setting away.

**Eq. 3.9 — pending decision.** `NOX_SOI_DIRECTION` below keeps the
text's formulation (`"decreasing"`). A2, Section 8 shows NOx *increasing*
with SOI inside the SOI block (r ≈ +0.99), in line with the prose of
Sec. 3.3.1.1 and the Papagiannakis example it cites. Once the SOI sign
convention is confirmed with the advisor, change this one setting (and
Eq. 3.9 / Table 7 in the text); both directions are unit-tested below.

In [ ]:
NOX_SOI_DIRECTION = "decreasing"   # Eq. 3.9 as written in the text -- see the pending-decision note above
PM_LAMBDA_DIRECTION = "decreasing"  # Eq. 3.10


def monotonic_constraint(predict_fn, x, out_idx, in_idx, direction):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        target = y[:, out_idx]
    grad = tape.gradient(target, x_t)
    d = grad[:, in_idx]
    violation = d if direction == "decreasing" else -d    # the derivative sign that is NOT allowed
    return tf.reduce_mean(tf.square(tf.maximum(0.0, violation)))

def nox_soi_constraint(predict_fn, x, direction=None):
    return monotonic_constraint(predict_fn, x, NOX_IDX, SOI_IDX, direction or NOX_SOI_DIRECTION)

def pm_lambda_constraint(predict_fn, x):
    return monotonic_constraint(predict_fn, x, PM_IDX, LAMBDA_IDX, PM_LAMBDA_DIRECTION)

print(f"NOx-SOI (Eq. 3.9), direction = {NOX_SOI_DIRECTION}:")
dec = lambda x: synthetic_output(x, {NOX_IDX: lambda x: -x[:, SOI_IDX]})   # NOx decreases with SOI
inc = lambda x: synthetic_output(x, {NOX_IDX: lambda x: x[:, SOI_IDX]})    # NOx increases with SOI
run_case(lambda f, x: nox_soi_constraint(f, x, "decreasing"), dec, X_test_pts,
         "decreasing setting, NOx decreasing", "zero", "NOx-SOI (3.9)")
run_case(lambda f, x: nox_soi_constraint(f, x, "decreasing"), inc, X_test_pts,
         "decreasing setting, NOx increasing", "positive", "NOx-SOI (3.9)")
run_case(lambda f, x: nox_soi_constraint(f, x, "increasing"), inc, X_test_pts,
         "increasing setting, NOx increasing", "zero", "NOx-SOI (3.9)")
run_case(lambda f, x: nox_soi_constraint(f, x, "increasing"), dec, X_test_pts,
         "increasing setting, NOx decreasing", "positive", "NOx-SOI (3.9)")

print("\nPM-lambda (Eq. 3.10):")
compliant = lambda x: synthetic_output(x, {PM_IDX: lambda x: -x[:, LAMBDA_IDX]})
violating = lambda x: synthetic_output(x, {PM_IDX: lambda x: x[:, LAMBDA_IDX]})
run_case(pm_lambda_constraint, compliant, X_test_pts, "PM decreasing", "zero", "PM-lambda (3.10)")
run_case(pm_lambda_constraint, violating, X_test_pts, "PM increasing", "positive", "PM-lambda (3.10)")

## 5. Shape constraint — HC vs. λ convexity (Eq. 3.11)

$$
\mathcal{L}_{\text{HC-}\lambda} = \frac{1}{N_c}\sum_{i=1}^{N_c} \max\!\left(0,\ -\frac{\partial^2 \text{HC}}{\partial \lambda^2}\Big|_i\right)^{2}
$$

**Second** derivative — needs a tape *inside* a tape: the inner tape
gets $\partial \text{HC}/\partial\lambda$, the outer tape differentiates
that result again.

In [ ]:
def convexity_constraint(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            y = predict_fn(x_t)
            target = y[:, out_idx]
        grad1 = tape1.gradient(target, x_t)
        d_first = grad1[:, in_idx]
    grad2 = tape2.gradient(d_first, x_t)
    d_second = grad2[:, in_idx]
    return tf.reduce_mean(tf.square(tf.maximum(0.0, -d_second)))

def hc_lambda_constraint(predict_fn, x):
    return convexity_constraint(predict_fn, x, HC_IDX, LAMBDA_IDX)

print("HC-lambda (Eq. 3.11):")
compliant = lambda x: synthetic_output(x, {HC_IDX: lambda x: (x[:, LAMBDA_IDX] - 0.5) ** 2})   # convex, U-shaped
violating = lambda x: synthetic_output(x, {HC_IDX: lambda x: -(x[:, LAMBDA_IDX] - 0.5) ** 2})  # concave
run_case(hc_lambda_constraint, compliant, X_test_pts, "HC convex (U-shaped)", "zero", "HC-lambda (3.11)")
run_case(hc_lambda_constraint, violating, X_test_pts, "HC concave", "positive", "HC-lambda (3.11)")

## 6. Trade-off constraints — NOx–PM (Eq. 3.12) and η–NOx (Eq. 3.13)

$$
\mathcal{L}_{\text{NOx-PM}} = \frac{1}{N_c}\sum_{i=1}^{N_c} \max\!\left(0,\ \frac{\partial \text{NOx}}{\partial \text{SOI}}\Big|_i \cdot \frac{\partial \text{PM}}{\partial \text{SOI}}\Big|_i\right)^{2}
$$

Both derivatives are with respect to **SOI** specifically (not a
generic input) — penalizes them moving in the *same* direction. Needs
**two** separate `tape.gradient()` calls from the same tape, so the
tape must be `persistent=True` (and explicitly deleted after, standard
practice to free its resources). η–NOx (Eq. 3.13) reuses the exact
same function with a different output pair; the text calls it a
*softer* constraint, meaning a smaller base weight and region-specific
validity (Section 8), not a different formula.

In [ ]:
def tradeoff_constraint(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a = y[:, out_idx_a]
        b = y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return tf.reduce_mean(tf.square(tf.maximum(0.0, grad_a * grad_b)))

def nox_pm_tradeoff_constraint(predict_fn, x):
    return tradeoff_constraint(predict_fn, x, NOX_IDX, PM_IDX, SOI_IDX)

def eff_nox_tradeoff_constraint(predict_fn, x):
    return tradeoff_constraint(predict_fn, x, ETA_IDX, NOX_IDX, SOI_IDX)

print("NOx-PM (Eq. 3.12):")
compliant = lambda x: synthetic_output(x, {NOX_IDX: lambda x: x[:, SOI_IDX], PM_IDX: lambda x: -x[:, SOI_IDX]})
violating = lambda x: synthetic_output(x, {NOX_IDX: lambda x: x[:, SOI_IDX], PM_IDX: lambda x: x[:, SOI_IDX]})
run_case(nox_pm_tradeoff_constraint, compliant, X_test_pts, "opposite directions", "zero", "NOx-PM (3.12)")
run_case(nox_pm_tradeoff_constraint, violating, X_test_pts, "same direction", "positive", "NOx-PM (3.12)")

print("\neta-NOx (Eq. 3.13):")
compliant = lambda x: synthetic_output(x, {ETA_IDX: lambda x: -x[:, SOI_IDX], NOX_IDX: lambda x: x[:, SOI_IDX]})
violating = lambda x: synthetic_output(x, {ETA_IDX: lambda x: x[:, SOI_IDX], NOX_IDX: lambda x: x[:, SOI_IDX]})
run_case(eff_nox_tradeoff_constraint, compliant, X_test_pts, "opposite directions", "zero", "eta-NOx (3.13)")
run_case(eff_nox_tradeoff_constraint, violating, X_test_pts, "same direction", "positive", "eta-NOx (3.13)")

## 7. Non-negativity (Eq. 3.14)

Eq. 3.14 asks for non-negative **physical** emissions. The network
predicts **normalized** values (Eq. 3.1, train-only min/max), where the
physical zero of emission $k$ sits at

$$
z_k^{0} = \frac{0 - x_{\min,k}}{x_{\max,k} - x_{\min,k}} \;<\; 0,
\qquad
\mathcal{L}_{\text{nonneg}} = \frac{1}{N}\sum_{i=1}^{N}\sum_{k \in \{\text{HC,NOx,CO2,PM}\}} \max\!\left(0,\ z_k^{0} - \hat{y}^{\,\text{norm}}_{i,k}\right)^{2}
$$

**Correction from the first version:** the penalty used to be
$\max(0, -\hat y^{\text{norm}})$, i.e. it penalized any prediction below
the *training minimum*, not below zero. That is a much stricter bound
than Eq. 3.14 — and it would have penalized correct predictions for the
validation/test points that A3 deliberately placed below the training
range. The table after the unit tests shows the floors $z_k^0$ and how
many measured val/test values lie between the floor and the training
minimum (the values the old version would have penalized).

Only these four outputs — **η is excluded**: Sec. 3.3.3.1 bounds it to
$(0,1)$ architecturally through the sigmoid head (A3, Section 9), so it
never needs this term. No gradient here, just the predicted values.

In [ ]:
to_fraction = (lambda v: v / 100) if df["eta"].max() > 1 else (lambda v: v)
EMISSIONS = [OUTPUT_COLS[i] for i in EMISSION_IDXS]
NONNEG_FLOOR_NORM = [float(-train_min[c] / (train_max[c] - train_min[c])) for c in EMISSIONS]


def nonneg_constraint(predict_fn, x, out_idxs=EMISSION_IDXS, floors=NONNEG_FLOOR_NORM):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    y = predict_fn(x_t)
    emissions = tf.gather(y, out_idxs, axis=1)
    floor_t = tf.constant(floors, dtype=tf.float32)          # normalized value of physical zero
    # Eq. 3.14: sum over the four emissions, mean over points
    return tf.reduce_mean(tf.reduce_sum(tf.square(tf.maximum(0.0, floor_t - emissions)), axis=1))


def const_rows(values):
    return {i: (lambda x, v=v: v + tf.zeros(tf.shape(x)[0])) for i, v in zip(EMISSION_IDXS, values)}

print("Non-negativity (Eq. 3.14):")
run_case(nonneg_constraint, lambda x: synthetic_output(x, const_rows([f + 0.5 for f in NONNEG_FLOOR_NORM])),
         X_test_pts, "above the physical zero", "zero", "non-negativity (3.14)")
run_case(nonneg_constraint, lambda x: synthetic_output(x, const_rows([f / 2 for f in NONNEG_FLOOR_NORM])),
         X_test_pts, "below train min, physically positive", "zero", "non-negativity (3.14)")
run_case(nonneg_constraint, lambda x: synthetic_output(x, const_rows([f - 0.5 for f in NONNEG_FLOOR_NORM])),
         X_test_pts, "physically negative", "positive", "non-negativity (3.14)")

**All unit tests** — every case from Sections 4–7 in one table. A failed
case is shown in the alert color and stops the notebook.

In [ ]:
unit_tests = pl.DataFrame(UNIT_TESTS).with_columns(pl.col("penalty").round(8))
flagged_table_html(unit_tests, "C1 -- Constraint unit tests on synthetic functions with known derivative sign",
                    HTML_DIR / "C1_07_constraint_unit_tests.html",
                    flag_col="passed", is_flagged=lambda v: not v)
assert all(unit_tests["passed"].to_list()), "a constraint unit test failed -- fix it before C2"
unit_tests

In [ ]:
non_train = df.filter(pl.col("split") != "train")
floor_rows = []
for c, z0 in zip(EMISSIONS, NONNEG_FLOOR_NORM):
    vals = non_train[f"{c}_norm"].to_numpy()
    floor_rows.append({
        "emission": c, "train_min": round(train_min[c], 5), "train_max": round(train_max[c], 5),
        "floor_norm (physical zero)": round(z0, 4),
        "val_test_below_train_min": int((vals < 0).sum()),
        "val_test_below_physical_zero": int((vals < z0).sum()),
    })
nonneg_floors = pl.DataFrame(floor_rows)
flagged_table_html(nonneg_floors, "C1 -- Non-negativity floors in normalized units (Eq. 3.14)",
                    HTML_DIR / "C1_07_nonneg_floors.html",
                    flag_col="val_test_below_physical_zero", is_flagged=lambda v: v > 0)
nonneg_floors

## 8. Region-specific validity & adaptive weighting (Eq. 3.15)

$$
\lambda_j(x) = \lambda_j^{0} \cdot \frac{\rho(x)}{\rho_{\max} + \epsilon} \cdot v_j(x)
$$

Three pieces, each computed from the **training data only**:

- $\rho(x)$ — local data density (Gaussian KDE over the 28 training
  points), normalized by its own max, so the constraint relaxes far
  from any experimental evidence.
- $v_j(x)$ — validity function. The text specifies $v_j(x)=1$
  everywhere for the four high-confidence constraints (NOx-SOI, PM-λ,
  HC-λ, NOx-PM). For η–NOx it depends on the local correlation between
  η and NOx in the training data.
- $\lambda_j^0$ — base weight per constraint, set in **C2** alongside
  the other loss weights, not here.

**η–NOx validity — which rule.** The text gives two readings:
Sec. 3.3.4.1 says $v_j \to 1$ where the *absolute* local correlation
exceeds 0.5, while Sec. 3.3.2.2 says the weight is reduced or set to
zero where the data shows no clear or a **positive** correlation. The
constraint penalizes η and NOx moving in the *same* direction, so with
the absolute value a strong positive correlation would switch it fully
**on** exactly where the data contradicts it. This notebook implements
the sign-aware rule, consistent with Sec. 3.3.2.2:

$$
v_{\eta\text{-NOx}}(x) = \operatorname{clip}\!\left(\frac{-r_{\text{local}}(x)}{0.5},\ 0,\ 1\right)
$$

— zero wherever the local correlation is positive, rising to 1 as it
reaches −0.5. The absolute-value version is computed alongside for
comparison. **Sec. 3.3.4.1 should be edited to match.**

**Implementation choice, not fully specified in the text:** "local"
correlation is a Gaussian-kernel-weighted Pearson correlation around each
query point (bandwidth 0.25 in normalized units) — a defensible reading,
but a reading, since Sec. 3.3.4.1 doesn't give the exact estimator.

In [ ]:
kde_train = gaussian_kde(X_train.T)

def density_ratio(x_query):
    rho = kde_train(x_query.T)
    return rho / (rho.max() + 1e-8) if rho.max() > 0 else rho

def local_correlation(x_query, x_ref, a_ref, b_ref, bandwidth=0.25):
    d = np.linalg.norm(x_ref[None, :, :] - x_query[:, None, :], axis=2)  # [n_query, n_ref]
    w = np.exp(-0.5 * (d / bandwidth) ** 2)
    w = w / (w.sum(axis=1, keepdims=True) + 1e-12)
    a_mean = (w * a_ref[None, :]).sum(axis=1)
    b_mean = (w * b_ref[None, :]).sum(axis=1)
    cov = (w * (a_ref[None, :] - a_mean[:, None]) * (b_ref[None, :] - b_mean[:, None])).sum(axis=1)
    var_a = (w * (a_ref[None, :] - a_mean[:, None]) ** 2).sum(axis=1)
    var_b = (w * (b_ref[None, :] - b_mean[:, None]) ** 2).sum(axis=1)
    return cov / np.sqrt(var_a * var_b + 1e-12)

VALIDITY_THRESHOLD = 0.5   # Sec. 3.3.4.1

eta_train = df.filter(pl.col("split") == "train")["eta_norm"].to_numpy()
nox_train = df.filter(pl.col("split") == "train")["NOx_norm"].to_numpy()

rho_colloc = density_ratio(X_colloc)
corr_colloc = local_correlation(X_colloc, X_train, eta_train, nox_train)
validity_eff_nox = np.clip(-corr_colloc / VALIDITY_THRESHOLD, 0, 1)            # implemented (Sec. 3.3.2.2)
validity_eff_nox_abs = np.clip(np.abs(corr_colloc) / VALIDITY_THRESHOLD, 0, 1)  # Sec. 3.3.4.1 wording

validity_comparison = pl.DataFrame([
    {"rule": name, "implemented": impl,
     "v_mean": round(float(v.mean()), 4), "v_min": round(float(v.min()), 4), "v_max": round(float(v.max()), 4),
     "share_active (v > 0)": round(float((v > 0).mean()), 4),
     "share_fully_active (v = 1)": round(float((v >= 0.999).mean()), 4)}
    for name, impl, v in [("signed: clip(-r / 0.5, 0, 1)  (Sec. 3.3.2.2)", True, validity_eff_nox),
                          ("absolute: clip(|r| / 0.5, 0, 1)  (Sec. 3.3.4.1)", False, validity_eff_nox_abs)]
])
simple_table_html(validity_comparison, "C1 -- eta-NOx validity over the 1000 collocation points: signed vs. absolute rule",
                   HTML_DIR / "C1_08_validity_comparison.html")
print(f"density ratio      : min={rho_colloc.min():.3f} max={rho_colloc.max():.3f}")
print(f"local corr(eta,NOx): min={corr_colloc.min():.3f} max={corr_colloc.max():.3f} "
      f"(share positive: {(corr_colloc > 0).mean():.1%})")
validity_comparison

**Distribution of the local correlation** that drives the η–NOx validity,
then both weighting components on the first two input dimensions (SOI vs.
λ, holding the LHS sample's own values for the other two):

In [ ]:
fig = go.Figure(go.Histogram(x=corr_colloc, nbinsx=40, marker_color=VARIABLE_COLORS["eta"], opacity=0.8))
for xv, txt in [(-VALIDITY_THRESHOLD, "v = 1 below this"), (0, "v = 0 above this")]:
    fig.add_vline(x=xv, line_dash="dash", line_color=SPLIT_COLORS["reference"], annotation_text=txt)
fig.update_layout(title="Local correlation r(eta, NOx) at the collocation points",
                  xaxis_title="kernel-weighted local Pearson r", yaxis_title="count", width=750, height=380)
fig.show()
fig.write_html(str(HTML_DIR / "C1_08_local_correlation_hist.html"), include_plotlyjs="inline")

fig = make_subplots(rows=1, cols=2, subplot_titles=["density ratio \u03c1(x)/\u03c1_max",
                                                    "validity v(x) for \u03b7-NOx (signed rule)"])
fig.add_trace(go.Scatter(x=X_colloc[:, SOI_IDX], y=X_colloc[:, LAMBDA_IDX], mode="markers",
                          marker=dict(color=rho_colloc, colorscale=[[0, "white"], [1, SPLIT_COLORS["reference"]]],
                                      size=5, showscale=True, colorbar=dict(x=0.46),
                                      line=dict(color=SPLIT_COLORS["neutral"], width=0.3)),
                          showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=X_colloc[:, SOI_IDX], y=X_colloc[:, LAMBDA_IDX], mode="markers",
                          marker=dict(color=validity_eff_nox, colorscale=[[0, "white"], [1, VARIABLE_COLORS["eta"]]],
                                      cmin=0, cmax=1, size=5, showscale=True,
                                      line=dict(color=SPLIT_COLORS["neutral"], width=0.3)),
                          showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="SOI (norm)")
fig.update_yaxes(title_text="lambda (norm)", col=1)
fig.update_layout(height=440, width=950, title_text="Adaptive weighting components over collocation points")
fig.show()
fig.write_html(str(HTML_DIR / "C1_08_adaptive_weighting.html"), include_plotlyjs="inline")

## 9. Wiring check — all six constraints on an untrained network

Not a meaningful physics result yet (random weights), just confirms
every constraint runs end-to-end against a real `keras` model — built
with the **same builder as B1/B2** (copied verbatim, including the η
sigmoid + `Rescaling` head) and the architecture **B1** selected — instead
of only the synthetic test harness. Every value must be finite and
non-negative; the notebook stops otherwise.

In [ ]:
import json
selection_path = OUT_DIR / "B1_selected_architecture.json"
if not selection_path.exists():
    raise FileNotFoundError(f"{selection_path} not found -- run B1 (including its save cell) before C1.")
with open(selection_path) as f:
    selection = json.load(f)
hidden_units = tuple(selection["hidden_units"])

assert OUTPUT_COLS[-1] == "eta", "the eta head is appended last; OUTPUT_COLS must end with eta"

# eta as a physical fraction (A1 Section 7: stored as a fraction) and its train-only range (A3 Section 9)
to_fraction = (lambda v: v / 100) if df["eta"].max() > 1 else (lambda v: v)
ETA_MIN, ETA_MAX = to_fraction(train_min["eta"]), to_fraction(train_max["eta"])
ETA_MEAN_TRAIN = float(np.mean(to_fraction(df.filter(pl.col("split") == "train")["eta"].to_numpy())))
ETA_BIAS_INIT = float(np.log(ETA_MEAN_TRAIN / (1 - ETA_MEAN_TRAIN)))    # logit(mean train eta)


def build_model(hidden_units, seed, input_dim=N_IN, output_dim=N_OUT):
    tf.random.set_seed(seed)
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for units in hidden_units:
        x = layers.Dense(units, activation="tanh")(x)
    emissions = layers.Dense(output_dim - 1, activation="linear", name="emissions")(x)   # HC, NOx, CO2, PM
    eta_frac = layers.Dense(1, activation="sigmoid", name="eta_fraction",
                            bias_initializer=keras.initializers.Constant(ETA_BIAS_INIT))(x)
    eta_norm = layers.Rescaling(scale=1.0 / (ETA_MAX - ETA_MIN),
                                offset=-ETA_MIN / (ETA_MAX - ETA_MIN), name="eta_rescaled")(eta_frac)
    outputs = layers.Concatenate(name="outputs")([emissions, eta_norm])
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="mse")
    return model


def predict_np(model, X):
    """Forward pass as a NumPy array, without model.predict().

    model.predict() builds a new tf.function for every freshly built model;
    in a loop of 240 fits that triggers TensorFlow's "tf.function retracing"
    warning and is slower than a direct call on arrays this small. A direct
    call with training=False gives the same predictions (no dropout or batch
    normalization in these models)."""
    return np.asarray(model(np.asarray(X, dtype="float32"), training=False))


probe_model = build_model(hidden_units, seed=SEED)
predict_fn = lambda x: probe_model(x, training=False)

results = {
    "NOx-SOI (3.9)": float(nox_soi_constraint(predict_fn, X_colloc)),
    "PM-lambda (3.10)": float(pm_lambda_constraint(predict_fn, X_colloc)),
    "HC-lambda (3.11)": float(hc_lambda_constraint(predict_fn, X_colloc)),
    "NOx-PM (3.12)": float(nox_pm_tradeoff_constraint(predict_fn, X_colloc)),
    "eta-NOx (3.13)": float(eff_nox_tradeoff_constraint(predict_fn, X_colloc)),
    "non-negativity (3.14)": float(nonneg_constraint(predict_fn, X_colloc)),
}
wiring_df = pl.DataFrame({"constraint": list(results.keys()),
                          "penalty": [float(f"{v:.6g}") for v in results.values()],
                          "finite_and_non_negative": [bool(np.isfinite(v) and v >= 0) for v in results.values()]})
flagged_table_html(wiring_df, f"C1 -- Physics penalties on an untrained {selection['architecture_id']} network",
                    HTML_DIR / "C1_09_wiring_table.html",
                    flag_col="finite_and_non_negative", is_flagged=lambda v: not v)
assert all(wiring_df["finite_and_non_negative"].to_list()), "a penalty is not finite / non-negative"
wiring_df

**As a picture** — no expected pattern yet (random weights), just confirms every value is finite and non-negative:

In [ ]:
fig = go.Figure(go.Bar(x=wiring_df["constraint"].to_list(), y=wiring_df["penalty"].to_list(),
                        marker_color=SPLIT_COLORS["neutral"],
                        marker_line=dict(color=SPLIT_COLORS["reference"], width=1)))
fig.update_layout(title="Physics penalties on an untrained network (sanity check only, log scale)",
                   yaxis=dict(title="penalty value", type="log"), width=750, height=420)
fig.show()
fig.write_html(str(HTML_DIR / "C1_09_wiring_bar.html"), include_plotlyjs="inline")

## Persist outputs

Saves the collocation points with the adaptive-weighting arrays (C2
doesn't have to regenerate the LHS sample or recompute the KDE/local
correlation) and a small configuration file with the choices made in
this notebook, so C2–C4 can check they use the same constraints.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
colloc_df = pl.DataFrame({c: X_colloc[:, i] for i, c in enumerate(INPUT_COLS)})
colloc_df = colloc_df.with_columns([
    pl.Series("density_ratio", rho_colloc),
    pl.Series("local_corr_eta_nox", corr_colloc),
    pl.Series("validity_eff_nox", validity_eff_nox),            # signed rule -- the one C2 uses
    pl.Series("validity_eff_nox_abs", validity_eff_nox_abs),    # Sec. 3.3.4.1 wording, for comparison only
])
colloc_df.write_csv(OUT_DIR / "C1_collocation_points.csv")

constraint_config = {
    "nox_soi_direction": NOX_SOI_DIRECTION,
    "pm_lambda_direction": PM_LAMBDA_DIRECTION,
    "nonneg_floor_norm": dict(zip(EMISSIONS, NONNEG_FLOOR_NORM)),
    "validity_eff_nox_rule": "clip(-r_local / 0.5, 0, 1)  (signed, Sec. 3.3.2.2)",
    "local_corr_bandwidth": 0.25,
    "n_colloc": N_COLLOC, "lhs_seed": SEED,
}
with open(OUT_DIR / "C1_constraint_config.json", "w") as f:
    json.dump(constraint_config, f, indent=2)
print(f"Saved to {OUT_DIR}")

## Next

**C2** assembles all six penalties plus the data loss and
regularization into the composite loss (Eq. 3.16), sets the base
weights $\lambda_j^0$, and calibrates them so no single term dominates
at initialization — this notebook only builds and tests the pieces,
C2 is where they get combined and weighted. C2 must use the constraint
functions exactly as defined here (including the physical-zero floors
and the signed η–NOx validity) and can check them against
`C1_constraint_config.json`.